# Week 2 · Day 4 — Supervised Learning II — Regression & Evaluation

**Course:** IPAM USL 5-Week Short Course: Introduction to Artificial Intelligence *(Introductory tier)*

**Facilitator:** Solomon Wilson MBCS | PhD Student, Computer Science | Deputy HOD Transport Planning & Operations | HOD, IT & Audit Supervisor, SLPTA

**Mode:** Google Colab (zero-install)

**Mental model layer:** L4 — Regression & Honest Evaluation

**Running scenario:** Route **R12** (Wilberforce → CBD) — operator OP-104, 25-minute delay

**Module:** 1 · **Week:** 2 · **Tier:** Intro  
**New concept:** Regression — predicting a continuous value (delay minutes) from features  
**Deliverable wired in:** None

## Learning objectives
By the end of today you will be able to:
- Train a model that predicts a **number** (how many minutes late) instead of a category.
- Read **MAE** and **R²**, and say what each means for dispatch.
- Recognise **overfitting** and use **cross-validation** to get an honest score.

## Why this matters for SLPTA

Yesterday we asked *"will this trip be badly late — yes or no?"* Today we ask the sharper question dispatch really wants: **"how many minutes late?"** Knowing R12 will likely run about 24 minutes behind is far more useful than just "late." Predicting a number is called **regression**.

## Environment setup

In [1]:
# Today's lab uses pandas + scikit-learn (both preinstalled in Colab). We also
# install google-genai so the SAME bootstrap import line works in every notebook.
!pip install -q google-genai
print("Environment ready.")

Environment ready.


In [2]:
# --- Standard SLPTA bootstrap (identical in every notebook) ----------------
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents,
                  Path("/content/IPAM_USL_Intro_AI_5Week")]:
    if (candidate / "shared" / "slpta_bootstrap.py").exists():
        sys.path.insert(0, str(candidate / "shared"))
        break

from slpta_bootstrap import (MODEL, ensure_course_data, get_client,
                             load_route12_context, load_route_logs,
                             load_complaints, load_routes, load_operators)

ensure_course_data()
print("Model configured:", MODEL)
print(load_route12_context())

Model configured: gemini-2.0-flash
Route R12 (Wilberforce → CBD). The 07:45 service, operated by OP-104 on vehicle SLPTA-1142, departed 25 minutes late. Recorded cause: Heavy traffic on Wilkinson Road. About 40 passengers were affected and the dispatch desk received multiple complaints. (Synthetic SLPTA scenario — no real data.)


## Concept — first principles

**Regression** predicts a continuous number. We measure error differently from classification:
- **MAE (Mean Absolute Error)** — on average, how many minutes our prediction is off. Lower is better, and it is in the same unit (minutes), so it is easy to explain to a manager.
- **R²** — the share of the variation in delay our features explain, from 0 (useless) to 1 (perfect).

Two traps we will meet today:
- **Overfitting** — a model that *memorises* the training trips but fails on new ones (low training error, high test error).
- To avoid being fooled by one lucky split, we use **cross-validation** — train and test on several different splits, then average.

<p align="center"></p>

<!-- cell-diagram:c09 -->
<p align="center"></p>

### Check your understanding (before running)
This cell loads Route R12 trip logs, removes outliers and duplicates, encodes features, and splits into train/test sets.

Before running:
- Which features do you think will matter most for predicting delay in minutes — route, weather, cause category, or time of day?
- After removing duplicates and delays ≥ 180 min, do you expect more or fewer than 1 500 rows to remain?

*Write your prediction, then run.*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# --- 1. Load and Deduplicate Data ---
# Retrieve raw route logs from the bootstrap utility
raw_logs = load_route_logs()
# Remove any duplicate records to ensure data integrity
logs = raw_logs.drop_duplicates()

# --- 2. Data Cleaning & Outlier Removal ---
# Filter out extreme outliers (delays >= 180 mins) to prevent skewing the model
logs = logs[logs["delay_minutes"] < 180]
# Create a copy to avoid SettingWithCopy warnings during imputation
logs = logs.copy()

# --- 3. Missing Value Imputation ---
# Determine the most frequent weather condition (mode)
weather_mode = logs["weather"].mode()[0]
# Fill missing weather values with the mode
logs["weather"] = logs["weather"].fillna(weather_mode)
# Determine the middle-point passenger count (median)
passenger_median = logs["passenger_count"].median()
# Fill missing passenger counts with the median
logs["passenger_count"] = logs["passenger_count"].fillna(passenger_median)

# --- 4. Feature Selection & Target Definition ---
# The target variable (y) is the continuous number of minutes late
y = logs["delay_minutes"]
# Select specific features relevant to the SLPTA regression task
feature_cols = ["route_id", "peak_period", "weather", "cause_category",
                "scheduled_hour", "distance_km", "passenger_count"]
features_df = logs[feature_cols]
# Convert categorical variables into numerical dummy indicators (One-Hot Encoding)
X = pd.get_dummies(features_df)

# --- 5. Data Splitting ---
# Split the dataset: 75% for training the model and 25% for testing its performance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

# --- 6. Baseline Performance Calculation ---
# Calculate the average delay from the training set to use as a naive predictor
train_mean_delay = y_train.mean()
# Calculate the Mean Absolute Error (MAE) if we simply guessed the average for every trip
baseline_mae = (y_test - train_mean_delay).abs().mean()

print(f"Baseline MAE (always guess the mean): {baseline_mae:.2f} minutes")

Baseline MAE (always guess the mean): 7.30 minutes


<!-- cell-diagram:c11 -->
<p align="center"></p>

### Check your understanding (before running)
**Predict:** a Linear Regression draws straight-line relationships; a Random Forest is far more flexible. Which one will predict delay minutes more accurately on **unseen** trips — or will the flexible one simply win?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# --- 1. Linear Regression Setup ---
# Initialize the Linear Regression estimator
linear_model = LinearRegression()
# Train (fit) the model using the training feature matrix and target vector
linear = linear_model.fit(X_train, y_train)
# Generate predictions for the unseen test data
linear_preds = linear.predict(X_test)

# --- 2. Random Forest Regressor Setup ---
# Initialize the ensemble model with 120 decision trees for robustness
forest_model = RandomForestRegressor(n_estimators=120, random_state=0)
# Train the forest model using the same training data
forest = forest_model.fit(X_train, y_train)
# Generate predictions for the unseen test data
forest_preds = forest.predict(X_test)

# --- 3. Performance Evaluation and Comparison ---
# Consolidate results into a list for systematic iteration
model_evaluations = [
    ("Linear Regression", y_test, linear_preds),
    ("Random Forest", y_test, forest_preds)
]

for name, actual_values, predicted_values in model_evaluations:
    # Calculate the average absolute deviation of predictions from actuals
    mae_score = mean_absolute_error(actual_values, predicted_values)
    # Calculate the proportion of variance explained by the model
    r2_val = r2_score(actual_values, predicted_values)

    # Print formatted output for the dispatch team to review
    print(f"{name:18s}  MAE={mae_score:.2f} min   R2={r2_val:.2f}")

Linear Regression   MAE=4.70 min   R2=0.58
Random Forest       MAE=5.29 min   R2=0.46


A surprise worth pausing on: the **simpler** Linear Regression is the more accurate here. Delays in our data are built mostly by *adding up* effects (peak + weather + cause), and a straight-line model fits that shape well. The lesson: a fancier model is not automatically better — match the model to the shape of the problem.

### Demo — watch the Random Forest overfit

<!-- cell-diagram:c15 -->
<p align="center"></p>

### Check your understanding (before running)
The Random Forest was trained on the training set. Before you run this cell:
- What do you expect the **training MAE** to be — high or low?
- What do you expect the **test MAE** to be compared to the training MAE?
- If the gap is large, what does that tell us about trusting this model for Route R12 predictions?

*Write your predictions, then run.*

In [ ]:
# --- Overfitting Analysis: Training vs. Testing Error ---

# 1. Evaluate performance on the training data (data the model has already seen)
# Generate predictions for the training set
forest_train_preds = forest.predict(X_train)
# Calculate Mean Absolute Error for the training set
train_mae = mean_absolute_error(y_train, forest_train_preds)

# 2. Evaluate performance on the test data (unseen data)
# Generate predictions for the test set
forest_test_preds = forest.predict(X_test)
# Calculate Mean Absolute Error for the test set
test_mae = mean_absolute_error(y_test, forest_test_preds)

# 3. Report findings
print(f"Random Forest  TRAIN MAE = {train_mae:.2f} min")
print(f"Random Forest  TEST MAE  = {test_mae:.2f} min")

# A significantly lower error on the training set compared to the test set
# indicates that the model has 'memorized' specific noise in the training
# data rather than learning generalizable patterns.
print("\nThe big gap means the forest partly MEMORIZED the training trips - that is overfitting.")

Random Forest  TRAIN MAE = 2.02 min
Random Forest  TEST MAE  = 5.29 min

The big gap means the forest partly MEMORIZED the training trips - that is overfitting.


<!-- cell-diagram:c17 -->
<p align="center"></p>

### Check your understanding (before running)
Cross-validation trains and tests on **5 different data splits**, then averages the R² scores.

Before running:
- Will the 5-fold R² be higher or lower than the single-split score we saw above?
- Why does averaging over multiple splits give a more honest picture of how this model would perform on real SLPTA trips?

*Write your prediction, then run.*

In [ ]:
from sklearn.model_selection import cross_val_score

# --- 1. Cross-Validation Configuration ---
# We use 5 folds (cv=5), meaning the data is split into 5 parts.
# The model trains on 4 and tests on 1, repeating this 5 times.
# We use 'r2' (Coefficient of Determination) to measure predictive power.
cv_scores = cross_val_score(linear, X, y, cv=5, scoring="r2")

# --- 2. Metric Aggregation ---
# Calculate the average R2 score across all 5 test folds
average_r2 = cv_scores.mean()
# Calculate the standard deviation to understand the consistency of the model
score_volatility = cv_scores.std()

# --- 3. Reporting ---
# Display the 'honest' score which represents expected real-world performance
print(f"Linear Regression 5-fold R2: {average_r2:.2f} (+/- {score_volatility:.2f})")

# Note: Averaging over multiple splits prevents a single 'lucky' or 'unlucky'
# train/test split from giving a misleading impression of model accuracy.
print("Averaging over 5 splits stops one lucky split from flattering the model.")

Linear Regression 5-fold R2: 0.60 (+/- 0.03)
Averaging over 5 splits stops one lucky split from flattering the model.


### Exercise — add and remove a feature

Change which feature we **drop**, re-run, and watch the error move. Dropping `passenger_count` barely changes anything; dropping `cause_category` should make R² collapse — that feature carries the most signal about why trips run late.

<!-- cell-diagram:c20 -->
<p align="center"></p>

### Check your understanding (before running)
This exercise drops one feature at a time and retrains the model. Before running:
- If we drop `cause_category`, do you expect MAE to go up or down? By a little or a lot?
- Which feature do you think carries the least signal about Route R12 delays — and why?

*Write your prediction, change the `feature_to_drop` string, then run.*

In [ ]:
# --- Exercise: Assessing Feature Importance by Removal ---

# Step 1: Select the feature you want to remove to test its impact
# Try "passenger_count", then "weather", then "cause_category"
feature_to_drop = "cause_category"

# Step 2: Define the full list of potential features
base_cols = ["route_id", "peak_period", "weather", "cause_category",
             "scheduled_hour", "distance_km", "passenger_count"]

# Step 3: Filter the list to exclude the selected feature
active_cols = [c for c in base_cols if c != feature_to_drop]

# Step 4: Prepare the feature matrix with One-Hot Encoding
features_subset = logs[active_cols]
X2 = pd.get_dummies(features_subset)

# Step 5: Split the new feature set into training and testing sets
Xtr, Xte, ytr, yte = train_test_split(X2, y, test_size=0.25, random_state=0)

# Step 6: Initialize and train the Linear Regression model
linear_reg = LinearRegression()
model = linear_reg.fit(Xtr, ytr)

# Step 7: Generate predictions and calculate performance metrics
predictions = model.predict(Xte)
mae_val = mean_absolute_error(yte, predictions)
r2_val = r2_score(yte, predictions)

# Step 8: Output the results for comparison
print(f"Dropped '{feature_to_drop}'  ->  Linear MAE = {mae_val:.2f} min   R2 = {r2_val:.2f}")

Dropped 'cause_category'  ->  Linear MAE = 6.49 min   R2 = 0.18


## Check your understanding
1. MAE is "minutes off on average." If a model's MAE is 5, what does that tell dispatch in plain words?
2. The Random Forest had a TRAIN MAE of about 2 but a TEST MAE of about 5. What is that gap called, and why is the TEST number the one we trust?
3. Which feature, when removed, hurt the prediction most? What does that say about what drives delay on the network?

### Your answers
*Double-click to edit this cell and type your answers here.*

1.
2.
3.

## If you remember one thing today…

> **Regression predicts a number, and the test-set error — not the training error — tells you whether to trust it.**

## Submission checklist
- [ ] Run every code cell successfully (top to bottom)
- [ ] Complete the exercise (fill every blank / make the requested change)
- [ ] Answer the **Check your understanding** questions in the markdown cell provided
- [ ] Save a clean copy of the notebook (*File → Save a copy in Drive*)